In [1]:
import os, re, sys
import io, queue, json, asyncio, httpx, numpy as np, sounddevice as sd, soundfile as sf
from openai import OpenAI
from torch import mode
from autogen_agentchat.agents import UserProxyAgent, AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient

In [2]:
# Project import paths
# Get the current directory (which is project/test)
notebook_dir = os.getcwd()

# Compute absolute paths for lib and src (one level up from test/)
lib_path = os.path.abspath(os.path.join(notebook_dir, '..', 'lib'))
src_path = os.path.abspath(os.path.join(notebook_dir, '..', 'src'))

sys.path.append(lib_path)
sys.path.append(src_path)

from model_loader import load_model_client

In [3]:

os.environ.setdefault("SPEACHES_BASE_URL", "http://localhost:8000/v1")
os.environ.setdefault("SPEACHES_API_KEY", "cant-be-empty")  # any non-empty if not enforcing auth
os.environ.setdefault("ASR_MODEL", "Systran/faster-distil-whisper-small.en")
os.environ.setdefault("ASR_LANGUAGE", "en")  # or leave "" to autodetect (slower)
os.environ.setdefault("RECORD_SECONDS", "3.8")

# Kokoro (via /v1/audio/speech)
os.environ.setdefault("TTS_MODEL", "speaches-ai/Kokoro-82M-v1.0-ONNX")  # or "kokoro"
os.environ.setdefault("TTS_VOICE", "af_heart")
os.environ.setdefault("TTS_PCM_RATE", "24000")  # Kokoro default


'24000'

In [4]:
SPEACHES_BASE = os.getenv("SPEACHES_BASE_URL").rstrip("/")
SPEACHES_KEY  = os.getenv("SPEACHES_API_KEY", "")
ASR_MODEL     = os.getenv("ASR_MODEL")
ASR_LANG      = (os.getenv("ASR_LANGUAGE", "") or "").strip() or None
REC_SECONDS   = float(os.getenv("RECORD_SECONDS", "3.8"))

TTS_MODEL     = os.getenv("TTS_MODEL")
TTS_VOICE     = os.getenv("TTS_VOICE")
TTS_RATE      = int(os.getenv("TTS_PCM_RATE", "24000"))

In [5]:
# ---- Mic recorder → BytesIO WAV (PCM_16), no disk ----
RATE, CH, BLOCK = 16000, 1, 1024  # 64ms blocks for stable capture

def record_bytes_wav(seconds: float = REC_SECONDS) -> io.BytesIO:
    q = queue.Queue()
    def cb(indata, frames, t, status):
        if status: print(status)
        q.put(indata.copy())

    with sd.InputStream(samplerate=RATE, channels=CH, blocksize=BLOCK, dtype="float32", callback=cb):
        needed = int(RATE / BLOCK * seconds)
        chunks = [q.get() for _ in range(max(1, needed))]
    audio = np.concatenate(chunks, axis=0)

    buf = io.BytesIO()
    sf.write(buf, audio, RATE, format="WAV", subtype="PCM_16")
    buf.seek(0)
    return buf

In [6]:
# ---- ASR: Speaches best-effort transcription ----

async def transcribe_best_effort(audio_wav) -> str:
    url = f"{SPEACHES_BASE}/audio/transcriptions"
    headers = {"Accept": "text/event-stream"}
    if SPEACHES_KEY:
        headers["Authorization"] = f"Bearer {SPEACHES_KEY}"

    files = {"file": ("mic.wav", audio_wav.getvalue(), "audio/wav")}

    # <-- paste the 'data = {...}' dict from Patch 1 here -->
    data = {
    "model": ASR_MODEL,
    "language": ASR_LANG or None,              # set "en" or "es" to skip autodetect
    "vad_filter": True,                       # VAD off (server default is on)
    "temperature": 0.2,                        # allow some exploration
    "temperature_increment_on_fallback": 0.2,  # try 0.2 → 0.4 if needed
    "log_prob_threshold": -1.2,                # be less strict than -1.0 default
    "no_speech_threshold": 0.3,                # less eager to call it silence
    "compression_ratio_threshold": None,       # don’t drop for CR
    }
    data = {k: v for k, v in data.items() if v is not None}

    final_text = ""
    # 1) Try streaming (tolerate SSE or raw JSON lines)
    try:
        async with httpx.AsyncClient(timeout=None) as ac:
            async with ac.stream("POST", url, params={"stream":"true"}, headers=headers, data=data, files=files) as resp:
                if resp.status_code == 200:
                    acc = ""
                    async for line in resp.aiter_lines():
                        if not line:
                            continue
                        # SSE case
                        if line.startswith("data:"):
                            payload = line[5:].strip()
                            if payload == "[DONE]":
                                break
                            try:
                                evt = json.loads(payload)
                                txt = evt.get("text") or evt.get("partial") or evt.get("data") or ""
                                if txt:
                                    print("🟢 partial:", txt)
                                    final_text = txt
                            except json.JSONDecodeError:
                                pass
                        else:
                            # raw JSON line (or chunk of it)
                            acc += line
                            try:
                                evt = json.loads(acc)
                                acc = ""  # consumed
                                txt = evt.get("text") or evt.get("partial") or evt.get("data") or ""
                                if txt:
                                    print("🟢 partial:", txt)
                                    final_text = txt
                            except json.JSONDecodeError:
                                # need more bytes
                                pass
                else:
                    # fall through to non-streaming
                    pass
    except Exception as e:
        print("SSE parse note:", e)

    if final_text:
        return final_text.strip()

    # 2) Fallback: non-streaming JSON (always returns something or error)
    async with httpx.AsyncClient(timeout=60.0) as ac:
        r = await ac.post(f"{SPEACHES_BASE}/audio/transcriptions",
                          headers={"Authorization": f"Bearer {SPEACHES_KEY}"} if SPEACHES_KEY else {},
                          data=data, files=files)
    r.raise_for_status()
    try:
        j = r.json()
        return (j.get("text") or j.get("transcript") or "").strip()
    except Exception:
        return (r.text or "").strip()


In [7]:
# ---- Speaches TTS streaming → speaker ----
tts_client = OpenAI(base_url=SPEACHES_BASE, api_key=SPEACHES_KEY)

def _frames_from_pcm16(b: bytes) -> np.ndarray:
    arr = np.frombuffer(b, dtype=np.int16)
    return arr.reshape(-1, 1) if arr.size else arr

def speak_stream(text: str):
    import sounddevice as sd
    tail = b""
    with sd.OutputStream(samplerate=TTS_RATE, channels=1, dtype="int16") as out:
        with tts_client.audio.speech.with_streaming_response.create(
            model=TTS_MODEL, voice=TTS_VOICE, input=text, response_format="pcm"
        ) as resp:
            for chunk in resp.iter_bytes(chunk_size=4096):
                if not chunk: continue
                data = tail + chunk
                cut  = (len(data)//2)*2
                play, tail = data[:cut], data[cut:]
                frames = _frames_from_pcm16(play)
                if frames.size: out.write(frames)
        if tail:
            out.write(_frames_from_pcm16(tail + b"\x00"))

In [8]:
# Use your preferred LLM here (OpenAI, local proxy, etc.)
model_client = load_model_client("azure")

In [9]:
VOICE_SYSTEM = (
    "You are the speaking part of a speech-to-speech assistant. "
    "Everything you say is spoken aloud. "
    "Reply in at most two short sentences. "
    "Do not include code blocks or links unless asked."
)

In [10]:
def _normalize_exit(s: str) -> str:
    s_clean = re.sub(r"[^\w\s]", "", (s or "")).strip().lower()
    return "EXIT" if s_clean in {"exit","quit","stop","bye","goodbye","salir"} else s

def _is_exit(s: str) -> bool:
    s = re.sub(r"[^\w\s]", "", (s or "")).strip().lower()
    return s in {"exit","quit","stop","bye","goodbye","salir"}

async def mic_input(_: str, __=None) -> str:
    print("\n🎙️  Speak now…")
    MAX_TRIES = 3
    for attempt in range(1, MAX_TRIES+1):
        buf = record_bytes_wav(REC_SECONDS)          # your in-memory recorder
        text = (await transcribe_best_effort(buf) or "").strip()
        print("🧾 final:", repr(text))
        if _is_exit(text):
            return "EXIT"
        if text:
            return text
        if attempt < MAX_TRIES:
            print("🤫 (heard silence/low-confidence) — try again…")
            await asyncio.sleep(0.2)
    # Still nothing — return a benign prompt instead of EXIT
    return "I didn’t catch that. Please repeat."


In [11]:

user = UserProxyAgent(name="user", input_func=mic_input)
assistant = AssistantAgent(name="assistant", system_message=VOICE_SYSTEM, model_client=model_client)

termination = TextMentionTermination("EXIT", sources=[user.name]) | MaxMessageTermination(2)
team = RoundRobinGroupChat([user, assistant], termination_condition=termination)

In [12]:
print("Voice chat ready. Say 'exit' to stop.\n")

# Main loop
while True:
    result = await team.run(task="Please speak your message.")
    last = next((m for m in reversed(result.messages)
                 if isinstance(m, TextMessage) and m.source == assistant.name), None)
    if last and last.content:
        try:
            speak_stream(last.content)
        except Exception as e:
            print("TTS error:", e)
    if any("EXIT" in (m.content or "") for m in result.messages):
        print("👋 Bye!")
        break

Voice chat ready. Say 'exit' to stop.


🎙️  Speak now…
🟢 partial: Hello?
🧾 final: 'Hello?'


/run/media/jun/Jun_6T/Code/TFM_BA_GPT/venv/lib/python3.13/site-packages/autogen_agentchat/agents/_assistant_agent.py:1107: UserWarning: Resolved model mismatch: gpt-4o-2024-08-06 != gpt-4o-2024-11-20. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to gpt-4o-2024-11-20 to enhance token/cost estimation and suppress this warning.
  model_result = await model_client.create(



🎙️  Speak now…
🟢 partial: Tell me about quantum
🧾 final: 'Tell me about quantum'

🎙️  Speak now…
🟢 partial: quantum neural networks.
🧾 final: 'quantum neural networks.'

🎙️  Speak now…
🟢 partial: X.
🧾 final: 'X.'

🎙️  Speak now…
🟢 partial: Exit
🧾 final: 'Exit'
👋 Bye!
